In [ ]:
import pandas as pd
from scipy.stats import chi2_contingency
from src.preprocess import load_train

# Categorical Feature Selection: Chi-Squared Test of Independence

This experiment evaluates the statistical significance of categorical features to prevent multicollinearity and remove noise from the final XGBoost pipeline.

**Statistical Hypotheses:**
* $H_0$ (Null Hypothesis): The feature and the target variable (Churn) are strictly independent. The feature contains no predictive signal.
* $H_A$ (Alternative Hypothesis): The feature and the target variable are dependent. The feature contains a predictive signal.

**Decision Rule:**
We evaluate the test statistic against a significance level of $\alpha = 0.05$. If the p-value $< 0.05$, we reject $H_0$ and retain the feature. If the p-value $> 0.05$, we fail to reject $H_0$ and drop the column.

In [ ]:
# 0. Features to test
feats = ['gender','PhoneService','MultipleLines']

# 1. Load the training data
X, y = load_train()

# 2. Re-attach the target variable temporarily for the mathematical comparison
evaluation_df = X.copy()
evaluation_df['Churn'] = y

for feat in feats:
    # 3. Create the observed contingency table
    contingency_table = pd.crosstab(evaluation_df[feat], evaluation_df['Churn'])

    # 4. Execute chi2 test
    chi2_stat, p_val, dof, expected = chi2_contingency(contingency_table)

    print(f'{feat} column')
    print(f"Chi-squared Statistic: {chi2_stat:.4f}")
    print(f"P-value: {p_val:.4e}")

    if p_val < 0.05:
        print(f"Reject H0: {feat} contains predictive signal. Keep the column")
    else:
        print(f"Fail to reject H0: {feat} and target are statistically independent. The column is meaningless")

    print('')

## As result, PhoneService and gender column will be dropped